# STAIR-Enhanced v3: Local Interest Aligned (STAIR-LIA)

## 📌 Giới thiệu Kiến trúc v3 (STAIR-LIA)
Phiên bản **STAIR-LIA v3** giải quyết triệt để các vấn đề của v1/v2a bằng 3 trụ cột kỹ thuật:
1. **Offline ZCA Whitening (Zero-phase):** Thay thế SVD để giữ nguyên hướng tọa độ gốc, bảo tồn cấu trúc phân rã chiều cho bộ làm mịn BSC Smoother.
2. **Offline Bidirectional Cross-Modal ROI Extraction (CLID-style):** Phân đoạn chuỗi ảo (6 text tokens, 64 visual patches) và tính Cross-Attention hai chiều hoàn toàn offline để lọc sạch nhiễu nền mà không gây xung đột gradient với BPR Loss.
3. **Denoised kNN Graph & Lightweight Fusion:** Xây dựng đồ thị $mAdj$ trực tiếp trên vector $E_{fused}$ sạch nhiễu và dung hợp qua hệ số $\alpha$ học nhẹ.

```
Text (384-D)   ──▶ [ ZCA Whitening ] ──▶ E_global_t (64-D) ──┐
Visual (4096-D) ──▶ [ ZCA Whitening ] ──▶ E_global_v (64-D) ──┼─▶ E_fused ─▶ Denoised kNN Graph (mAdj)
                       ▲                                      │
                       └─ [ Cross-Modal Attention ROI ] ──────┘
                          E_roi_t (64-D), E_roi_v (64-D)
```


## Cell 1 — Thiết lập Môi trường & Cài đặt STAIR-Enhanced


In [ ]:
# Cell 1: Môi trường & Cài đặt Dependencies
import sys, os, subprocess, shutil, torch

print(f'Python version : {sys.version}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total     : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')

# Cài đặt các gói bổ trợ
for pkg in ['prettytable', 'pynvml']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

REPO_URL = 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'
WORK_DIR = '/kaggle/working/STAIR-Enhanced'

if os.path.exists(WORK_DIR):
    print(f'Updating existing STAIR-Enhanced repository...')
    subprocess.run(['git', 'pull'], cwd=WORK_DIR, check=True)
else:
    print(f'Cloning STAIR-Enhanced...')
    subprocess.run(['git', 'clone', REPO_URL, WORK_DIR], check=True)

sys.path.insert(0, WORK_DIR)
print('[OK] STAIR-Enhanced v3 LIA sẵn sàng!')


## Cell 2 — Chuẩn bị Dữ liệu từ Kaggle Input (Tự động quét linh hoạt)


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data & STAIR-Enhanced/data
# Sử dụng os.walk linh hoạt để phát hiện dữ liệu ở bất kỳ slug / thư mục con nào
import os, shutil

DATA_ROOTS = ['/kaggle/data', '/kaggle/working/STAIR-Enhanced/data']
for root in DATA_ROOTS:
    os.makedirs(root, exist_ok=True)

REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt')

def copy_dataset(keywords, full_name):
    copied_files = 0
    for root_dir, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root_dir.lower() for kw in keywords):
            for f in files:
                if f.endswith(REQUIRED_EXTENSIONS):
                    src_path = os.path.join(root_dir, f)
                    for target_root in DATA_ROOTS:
                        dest_dir = os.path.join(target_root, full_name)
                        os.makedirs(dest_dir, exist_ok=True)
                        shutil.copy(src_path, os.path.join(dest_dir, f))
                    copied_files += 1
    print(f'[OK] {full_name}: {copied_files} files copied.')

copy_dataset(['baby',        'amazon2014baby'],        'Amazon2014Baby_550_MMRec')
copy_dataset(['sports',      'amazon2014sports'],      'Amazon2014Sports_550_MMRec')
copy_dataset(['electronics', 'amazon2014electronics'], 'Amazon2014Electronics_550_MMRec')

print(f'\nDữ liệu sẵn sàng tại: {DATA_ROOTS[0]}')


## Cell 3 — Tiền xử lý Offline: ZCA Whitening + ROI Attention (STAIR-LIA)


In [ ]:
# Cell 3: Chạy Tiền xử lý Offline (ZCA Whitening + Bidirectional Cross-Modal ROI)
# Chạy 1 lần duy nhất trên GPU, trích xuất E_fused_t.pt, E_fused_v.pt lưu vào đĩa
import os, sys, subprocess, time

PREPROC_SCRIPT = '/kaggle/working/STAIR-Enhanced/preprocess_stair_lia.py'
PREPROC_OUT    = '/kaggle/working/preprocessed_lia'
os.makedirs(PREPROC_OUT, exist_ok=True)

DATASETS = [
    ('Amazon2014Baby_550_MMRec',        'baby'),
    ('Amazon2014Sports_550_MMRec',      'sports'),
    ('Amazon2014Electronics_550_MMRec', 'electronics'),
]

for ds_name, key in DATASETS:
    ds_dir = f'/kaggle/data/{ds_name}'
    if not os.path.exists(ds_dir):
        continue
    
    # Tìm file text và visual features
    text_f, vis_f = None, None
    for fname in os.listdir(ds_dir):
        fn_l = fname.lower()
        if 'text' in fn_l and fname.endswith(('.pkl', '.npy', '.pt')):
            text_f = os.path.join(ds_dir, fname)
        if ('vis' in fn_l or 'image' in fn_l or 'img' in fn_l) and fname.endswith(('.pkl', '.npy', '.pt')):
            vis_f = os.path.join(ds_dir, fname)
            
    if text_f and vis_f:
        print(f'\n[Preprocessing {key.upper()}]')
        print(f'  Text  : {text_f}')
        print(f'  Visual: {vis_f}')
        t0 = time.time()
        cmd = [
            sys.executable, PREPROC_SCRIPT,
            '--dataset',      ds_name,
            '--text_feat',    text_f,
            '--visual_feat',  vis_f,
            '--output_dir',   PREPROC_OUT,
            '--fuweight',     '0.6',
            '--device',       'cuda' if torch.cuda.is_available() else 'cpu',
            '--sanity_check',
        ]
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode == 0:
            print(f'  [OK] Hoàn tất sau {time.time()-t0:.1f}s')
            # Copy kết quả vào thư mục dữ liệu để main nạp tự động
            src_out = os.path.join(PREPROC_OUT, ds_name)
            for target_root in DATA_ROOTS:
                dst_out = os.path.join(target_root, ds_name, 'preprocessed_lia', ds_name)
                os.makedirs(dst_out, exist_ok=True)
                for item in os.listdir(src_out):
                    shutil.copy(os.path.join(src_out, item), os.path.join(dst_out, item))
        else:
            print(f'  [LỖI]: {res.stderr[-500:]}')

print('\n[Cell 3 DONE] Tiền xử lý offline hoàn tất!')


## Cell 4 — Kiểm tra Modules STAIR-LIA v3 & Tensors trước khi chạy


In [ ]:
# Cell 4: Kiểm tra cấu hình và Modules STAIR-LIA v3
import torch
from main_stair_lia_v3 import STAIR_LIA_v3, roi_contrastive_loss

print('[1/2] Module STAIR_LIA_v3 import thành công!')

# Kiểm tra hàm ROI Contrastive Loss
t_dummy = torch.randn(32, 64)
v_dummy = torch.randn(32, 64)
cl_val = roi_contrastive_loss(t_dummy, v_dummy, temperature=0.07)
print(f'[2/2] Test ROI Contrastive Loss: loss_val={cl_val.item():.4f}')

print('\n[OK] Toàn bộ module STAIR-LIA v3 đã sẵn sàng huấn luyện!')


## Cell 5 — Helper Functions & Training Runner (Kèm VRAM Profiler)


In [ ]:
# Cell 5: Hàm hỗ trợ chạy Training & Giám sát Phần cứng
import subprocess, threading, time, os, re, pynvml

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    pynvml.nvmlInit()
    h = pynvml.nvmlDeviceGetHandleByIndex(0)
    records = []
    while not stop_evt.is_set():
        mem = pynvml.nvmlDeviceGetMemoryInfo(h)
        records.append(mem.used / 1024**2)
        time.sleep(interval)
    pynvml.nvmlShutdown()
    vram_profile[key] = records

def extract_best_test(log_path):
    if not os.path.exists(log_path):
        return None, None
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    best_epoch = None
    best_metrics = {}
    for line in lines:
        if 'TEST @Epoch' in line:
            ep_match = re.search(r'TEST @Epoch:\s*(\d+)', line)
            if ep_match:
                best_epoch = int(ep_match.group(1))
        if any(k in line for k in ['Recall@20', 'NDCG@20', 'Recall@10', 'NDCG@10']):
            for metric in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
                m = re.search(rf'{metric}\s*Avg:\s*([0-9.]+)', line, re.IGNORECASE)
                if m:
                    best_metrics[metric] = float(m.group(1))
    return best_epoch, best_metrics

def parse_alpha_history(log_path):
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'alpha @epoch\s*(\d+).*?:\s*([0-9.]+)', content)
    return [(int(ep), float(val)) for ep, val in matches]

def run_training(key, yaml_cfg, data_root, log_path, lia_alpha=0.5, lia_roi_cl=0.01):
    print('=' * 60)
    print(f'BẮT ĐẦU HUẤN LUYỆN v3 (STAIR-LIA): {key.upper()}')
    print(f'Config : {yaml_cfg}')
    print(f'Log    : {log_path}')
    print('=' * 60)
    
    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()
    
    t0 = time.time()
    cmd = [
        sys.executable, '/kaggle/working/STAIR-Enhanced/main_stair_lia_v3.py',
        '--config', yaml_cfg,
        '--root',   data_root,
        '--lia-alpha',  str(lia_alpha),
        '--lia-roi-cl', str(lia_roi_cl),
    ]
    with open(log_path, 'w', encoding='utf-8') as f:
        result = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, cwd='/kaggle/working/STAIR-Enhanced')
    
    elapsed = time.time() - t0
    stop_evt.set()
    th.join(timeout=3)
    
    if result.returncode != 0:
        print(f'[THẤT BẠI] Mã lỗi {result.returncode} (Thời gian: {elapsed/60:.1f} phút)')
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            print('\n'.join(f.readlines()[-25:]))
    else:
        print(f'[HOÀN THÀNH] {key.upper()} trong {elapsed/60:.1f} phút')
        ep, metrics = extract_best_test(log_path)
        if metrics:
            print(f'  - Best checkpoint @Epoch: {ep}')
            for k, v in metrics.items():
                print(f'    * {k}: {v:.6f}')
    return result.returncode


## Cell 6 — Huấn luyện STAIR-LIA v3 trên Baby & Sports


In [ ]:
# Cell 6: Huấn luyện STAIR-LIA v3 trên Baby & Sports
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_lia_v3'
os.makedirs(LOG_DIR, exist_ok=True)

# 1. Huấn luyện Baby
run_training(
    key       = 'baby',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Baby_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/baby.log',
    lia_alpha = 0.5,
    lia_roi_cl= 0.01,
)
torch.cuda.empty_cache()

# 2. Huấn luyện Sports
run_training(
    key       = 'sports',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Sports_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/sports.log',
    lia_alpha = 0.5,
    lia_roi_cl= 0.01,
)
torch.cuda.empty_cache()

print('=' * 60)
print('✅ HOÀN THÀNH HUẤN LUYỆN BABY & SPORTS (STAIR-LIA v3)!')
print('=' * 60)


## Cell 7 — Huấn luyện STAIR-LIA v3 trên Electronics (Tập lớn nhất ~1.7M tương tác)


In [ ]:
# Cell 7: Huấn luyện STAIR-LIA v3 trên Electronics
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_lia_v3'
os.makedirs(LOG_DIR, exist_ok=True)

run_training(
    key       = 'electronics',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Electronics_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/electronics.log',
    lia_alpha = 0.5,
    lia_roi_cl= 0.01,
)
torch.cuda.empty_cache()

print('=' * 60)
print('✅ HOÀN THÀNH HUẤN LUYỆN ELECTRONICS (STAIR-LIA v3)!')
print('=' * 60)


## Cell 8 — Tổng hợp Kết quả & So sánh Ablation Study 4 Phiên bản


In [ ]:
# Cell 8: Bảng so sánh Ablation Study: Baseline vs v1 vs v2a vs v3 (LIA)
from prettytable import PrettyTable
import os, math

LOG_DIR_V3 = '/kaggle/working/logs_lia_v3'

BASELINE = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V1_RESULTS = {
    'baby':        {'Recall@10': 0.0611, 'Recall@20': 0.0948, 'NDCG@10': 0.0325, 'NDCG@20': 0.0412},
    'sports':      {'Recall@10': 0.0695, 'Recall@20': 0.1040, 'NDCG@10': 0.0376, 'NDCG@20': 0.0466},
    'electronics': {'Recall@10': 0.0401, 'Recall@20': 0.0601, 'NDCG@10': 0.0223, 'NDCG@20': 0.0274},
}

V2A_RESULTS = {
    'baby':        {'Recall@10': 0.0663, 'Recall@20': 0.1026, 'NDCG@10': 0.0351, 'NDCG@20': 0.0445},
    'sports':      {'Recall@10': 0.0738, 'Recall@20': 0.1102, 'NDCG@10': 0.0401, 'NDCG@20': 0.0494},
    'electronics': {'Recall@10': 0.0435, 'Recall@20': 0.0658, 'NDCG@10': 0.0241, 'NDCG@20': 0.0298},
}

v3_results = {}
for ds in ['baby', 'sports', 'electronics']:
    log = os.path.join(LOG_DIR_V3, f'{ds}.log')
    ep, metrics = extract_best_test(log)
    v3_results[ds] = {'epoch': ep, 'metrics': metrics}

METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

print('=' * 110)
print('BẢNG SO SÁNH ABLATION STUDY: STAIR Baseline vs Enhanced v1 vs v2a vs v3 (STAIR-LIA)')
print('=' * 110)

for ds in ['baby', 'sports', 'electronics']:
    t = PrettyTable()
    t.field_names = ['Chỉ số', 'STAIR Baseline', 'v1 (Replace)', 'v2a (Dynamic Res)', 'v3 (STAIR-LIA)', 'Δ(v3 vs Baseline)']
    t.align = 'r'; t.align['Chỉ số'] = 'l'
    bl  = BASELINE[ds]
    v1  = V1_RESULTS[ds]
    v2a = V2A_RESULTS[ds]
    v3  = v3_results[ds]['metrics'] or {}
    for m in METRICS:
        bl_val  = bl.get(m, float('nan'))
        v1_val  = v1.get(m, float('nan'))
        v2a_val = v2a.get(m, float('nan'))
        v3_val  = v3.get(m, float('nan'))
        delta   = f"{(v3_val - bl_val)/bl_val*100:+.2f}%" if (bl_val and v3_val and not math.isnan(v3_val)) else 'N/A'
        t.add_row([
            m,
            f'{bl_val:.4f}',
            f'{v1_val:.4f}' if v1_val else 'N/A',
            f'{v2a_val:.4f}' if v2a_val else 'N/A',
            f'{v3_val:.4f}' if (v3_val and not math.isnan(v3_val)) else 'N/A',
            delta
        ])
    print(f'\nTập dữ liệu: {ds.upper()} (Best Epoch v3 LIA: {v3_results[ds]["epoch"]})')
    print(t)
print('=' * 110)


## Cell 9 — Biểu đồ Quỹ đạo Alpha & Đường cong Học tập


In [ ]:
# Cell 9: Vẽ biểu đồ Alpha Evolution & VRAM Profiling
import matplotlib.pyplot as plt, os

LOG_DIR = '/kaggle/working/logs_lia_v3'
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('STAIR-LIA v3: Alpha (Text Modality Weight) Evolution qua các Epochs', fontsize=14, fontweight='bold')

for i, ds in enumerate(['baby', 'sports', 'electronics']):
    log_path = os.path.join(LOG_DIR, f'{ds}.log')
    history = parse_alpha_history(log_path)
    ax = axes[i]
    if history:
        epochs  = [h[0] for h in history]
        alphas  = [h[1] for h in history]
        ax.plot(epochs, alphas, 'b-', linewidth=2, label='alpha (text weight)')
        ax.plot(epochs, [1 - a for a in alphas], 'r--', linewidth=1.5, label='1-alpha (visual weight)')
        ax.axhline(y=0.5, color='gray', linestyle=':', label='Initial (0.5)')
        ax.set_ylim(0, 1)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Weight')
        ax.set_title(f'{ds.capitalize()}')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Chưa có log', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{ds.capitalize()} (No data)')

plt.tight_layout()
plt.savefig('/kaggle/working/alpha_evolution_v3_lia.png', dpi=150)
plt.show()
print('[ĐÃ LƯU BIỂU ĐỒ] /kaggle/working/alpha_evolution_v3_lia.png')


## Cell 10 — Xuất Bảng Kết quả CSV cho Khóa luận Tốt nghiệp


In [ ]:
# Cell 10: Xuất bảng kết quả CSV
import csv, os

OUT_CSV = '/kaggle/working/ablation_enhanced_v3_lia.csv'
METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

rows = []
for ds in ['baby', 'sports', 'electronics']:
    bl  = BASELINE[ds]
    v1  = V1_RESULTS[ds]
    v2a = V2A_RESULTS[ds]
    v3  = v3_results[ds]['metrics'] or {}
    ep  = v3_results[ds]['epoch']
    for m in METRICS:
        bl_v = bl.get(m)
        v1_v = v1.get(m)
        v2_v = v2a.get(m)
        v3_v = v3.get(m)
        delta_v3 = (v3_v - bl_v)/bl_v*100 if (bl_v and v3_v) else None
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'STAIR-Baseline', 'value': f'{bl_v:.4f}' if bl_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v1 (Replace)', 'value': f'{v1_v:.4f}' if v1_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v2a (Dynamic)', 'value': f'{v2_v:.4f}' if v2_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v3 (STAIR-LIA)', 'value': f'{v3_v:.4f}' if v3_v else 'N/A', 'best_epoch': str(ep) if ep else 'N/A'})
        if delta_v3 is not None:
            rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Delta v3 vs Baseline (%)', 'value': f'{delta_v3:+.2f}%', 'best_epoch': ''})

with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'metric', 'model', 'value', 'best_epoch'])
    writer.writeheader()
    writer.writerows(rows)

print(f'[ĐÃ XUẤT CSV] {OUT_CSV}')
